# Prediction of the target in the Test.csv

## Preprocessing of the Test.csv

In [6]:
# ============================================================
# NOTEBOOK 4 — FINAL MODEL TRAINING & SUBMISSION GENERATION
# ============================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import pickle

from preprocessing_mel import preprocess_minimal   # your minimal leakage-safe preprocessor

RSEED = 42

# ============================================================
# 1. Load and split Train.csv
# ============================================================
df_train = pd.read_csv("data/Train.csv")
X_full = df_train.drop(columns=['target'])
y_full = df_train['target']

# ============================================================
# 2. Fit preprocessing on full training data
# ============================================================
X_full_proc, y_full_proc = preprocess_minimal(
    df=X_full,
    y=y_full,
    fit_data=X_full  # learn imputer/scaler only from train
)

print("Processed train shape:", X_full_proc.shape)

# ============================================================
# 3. Apply preprocessing to Test.csv (NO FITTING)
# ============================================================
df_test = pd.read_csv("data/Test.csv")
X_test_proc = preprocess_minimal(
    df=df_test,
    fit_data=X_full  # use training statistics
)

print("Processed test shape:", X_test_proc.shape)

# ============================================================
# 4. Train final model on full train and predict on Test.csv
# ============================================================
from sklearn.ensemble import RandomForestRegressor

rf_final = RandomForestRegressor(
    n_estimators=400,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    max_depth=16,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X_full_proc, y_full_proc)
y_test_pred = rf_final.predict(X_test_proc)

# ============================================================
# 5. Prepare submission
# ============================================================
df_submission = pd.DataFrame({
    "Place_ID X Date": df_test["Place_ID X Date"],
    "target": y_test_pred
})

df_submission.to_csv("submission.csv", index=False)
print("✅ Submission saved!")


Processed train shape: (30557, 74)
Processed test shape: (16136, 74)
✅ Submission saved!
